In [ ]:
# ══════════════════════════════════════════════════════════════════════
# H11 — BSDT Probabilistic Repair V2 (BPR)
# ══════════════════════════════════════════════════════════════════════
# Novel SAT local-search repair:
#
#   WalkSAT (Selman 1994):  p(flip v) based on break count (integer)
#   probSAT (Balint 2012):  p(flip v) ∝ break(v)^(-cb) (integer poly)
#   BPR     (Odeyemi 2026): p(flip v) ∝ exp(-(make-break)/T) × confidence(v)
#
# Key innovation vs ALL prior SLS solvers:
#   "confidence(v)" = 1 - |s_v| where s_v ∈ [-1,1] is the continuous
#   BSDT particle position BEFORE rounding. Variables near 0 in the
#   continuous landscape were AMBIGUOUS — flipping them is cheap.
#   Variables near ±1 were CONFIDENT — flipping them is expensive.
#   No other solver has this continuous prior.
#
# Optimisations:
#   1. Incremental ΔE from clause_sat (no flip/unflip) — O(degree(v))
#   2. Incremental unsat list maintained across flips — O(1) random pick
#   3. Flat-array var→clause adjacency — Numba-friendly
#   4. Confidence weights pre-computed once from continuous positions
#
# Author: Odeyemi Olusegun Israel
# ══════════════════════════════════════════════════════════════════════
import torch, numpy as np, time
from numba import njit

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# ── Instance generator ────────────────────────────────────────────────
def generate_3sat_instance(n, m):
    vars_idx = torch.randint(0, n, (m, 3))
    signs = torch.randint(0, 2, (m, 3)) * 2 - 1
    return list(zip(vars_idx.tolist(), signs.tolist()))

# ══════════════════════════════════════════════════════════════════════
#  BPR — BSDT Probabilistic Repair (fully optimised Numba)
# ══════════════════════════════════════════════════════════════════════

@njit(cache=True)
def bpr_repair(clauses_v, clauses_s, assignment, confidence,
               max_flips=100000, T_init=0.5, T_min=0.01, p_random=0.1):
    """
    BSDT Probabilistic Repair — energy-guided + confidence-weighted.

    Parameters
    ----------
    clauses_v : (m, 3) int32  — variable indices (0-indexed)
    clauses_s : (m, 3) int32  — signs (+1 / -1)
    assignment : (n,) int32   — initial 0/1 assignment (modified in-place)
    confidence : (n,) float64 — 1 - |s_v| from continuous BSDT positions
                                high = ambiguous (cheap to flip)
                                low = confident (expensive to flip)
    max_flips : int
    T_init, T_min : float     — Boltzmann temperature (cosine annealed)
    p_random : float          — exploration probability

    Returns
    -------
    assignment, flips_used
    """
    m = clauses_v.shape[0]
    n = assignment.shape[0]

    # ── Build flat var→clause adjacency ──
    var_count = np.zeros(n, dtype=np.int32)
    for c in range(m):
        for j in range(3):
            var_count[clauses_v[c, j]] += 1
    var_off = np.zeros(n + 1, dtype=np.int32)
    for v in range(n):
        var_off[v + 1] = var_off[v] + var_count[v]
    var_adj = np.zeros(var_off[n], dtype=np.int32)
    fill = np.zeros(n, dtype=np.int32)
    for c in range(m):
        for j in range(3):
            v = clauses_v[c, j]
            var_adj[var_off[v] + fill[v]] = c
            fill[v] += 1

    # ── Build var→sign-in-clause lookup ──
    # For each (var, clause) pair, what sign does var have in that clause?
    var_sign = np.zeros(var_off[n], dtype=np.int32)
    fill2 = np.zeros(n, dtype=np.int32)
    for c in range(m):
        for j in range(3):
            v = clauses_v[c, j]
            var_sign[var_off[v] + fill2[v]] = clauses_s[c, j]
            fill2[v] += 1

    # ── Init clause satisfaction counts ──
    clause_sat = np.zeros(m, dtype=np.int32)
    for c in range(m):
        for j in range(3):
            v = clauses_v[c, j]
            s = clauses_s[c, j]
            if (assignment[v] == 1 and s == 1) or (assignment[v] == 0 and s == -1):
                clause_sat[c] += 1

    # ── Incremental unsatisfied clause list ──
    # unsat_list[0..n_unsat-1] = clause indices that are unsatisfied
    # unsat_pos[c] = position of clause c in unsat_list (-1 if satisfied)
    unsat_list = np.zeros(m, dtype=np.int32)
    unsat_pos = np.full(m, -1, dtype=np.int32)
    n_unsat = 0
    for c in range(m):
        if clause_sat[c] == 0:
            unsat_pos[c] = n_unsat
            unsat_list[n_unsat] = c
            n_unsat += 1

    for flip in range(max_flips):
        if n_unsat == 0:
            return assignment, flip

        # ── Pick random unsatisfied clause O(1) ──
        ci = unsat_list[np.random.randint(n_unsat)]

        # ── Temperature (cosine anneal) ──
        progress = flip / max_flips
        T = T_min + (T_init - T_min) * 0.5 * (1.0 + np.cos(3.141592653589793 * progress))

        if np.random.random() < p_random:
            # Pure random exploration
            v_flip = clauses_v[ci, np.random.randint(3)]
        else:
            # ── Compute make/break + confidence for each candidate ──
            scores = np.zeros(3, dtype=np.float64)

            for j in range(3):
                v_cand = clauses_v[ci, j]
                s_in_clause = clauses_s[ci, j]

                # Does v_cand currently satisfy clause ci?
                v_satisfies = ((assignment[v_cand] == 1 and s_in_clause == 1) or
                               (assignment[v_cand] == 0 and s_in_clause == -1))

                make = 0
                brk = 0

                # Scan all clauses containing v_cand
                for idx in range(var_off[v_cand], var_off[v_cand + 1]):
                    cc = var_adj[idx]
                    s_here = var_sign[idx]

                    # Does v_cand currently satisfy clause cc?
                    satisfies_cc = ((assignment[v_cand] == 1 and s_here == 1) or
                                    (assignment[v_cand] == 0 and s_here == -1))

                    if satisfies_cc:
                        # Flipping removes this satisfaction
                        if clause_sat[cc] == 1:
                            brk += 1  # clause becomes UNSAT
                    else:
                        # Flipping adds satisfaction
                        if clause_sat[cc] == 0:
                            make += 1  # clause becomes SAT

                # ── BPR score = energy delta + confidence ──
                # ΔE ∝ (break - make): positive = bad, negative = good
                # confidence[v] = 1 - |s_v|: high = ambiguous = cheap to flip
                #
                # score = exp(-(break - make) / T) × (confidence[v] + 0.1)
                #
                # The confidence term is the NOVEL part:
                # all prior SLS use only (break, make). We add continuous info.
                delta = brk - make
                conf = confidence[v_cand] + 0.1  # floor of 0.1 prevents zero
                scores[j] = np.exp(-delta / (T + 1e-10)) * conf

            # ── Sample from Boltzmann × confidence distribution ──
            total = scores[0] + scores[1] + scores[2]
            if total < 1e-30:
                # Degenerate — pick random
                v_flip = clauses_v[ci, np.random.randint(3)]
            else:
                r = np.random.random() * total
                if r <= scores[0]:
                    v_flip = clauses_v[ci, 0]
                elif r <= scores[0] + scores[1]:
                    v_flip = clauses_v[ci, 1]
                else:
                    v_flip = clauses_v[ci, 2]

        # ── Execute flip + incremental updates O(degree(v)) ──
        assignment[v_flip] = 1 - assignment[v_flip]

        for idx in range(var_off[v_flip], var_off[v_flip + 1]):
            cc = var_adj[idx]
            s_here = var_sign[idx]
            old_sat = clause_sat[cc]

            # After flip: does v_flip now satisfy clause cc?
            now_satisfies = ((assignment[v_flip] == 1 and s_here == 1) or
                             (assignment[v_flip] == 0 and s_here == -1))

            if now_satisfies:
                clause_sat[cc] += 1
            else:
                clause_sat[cc] -= 1

            new_sat = clause_sat[cc]

            # Update unsat list incrementally
            if old_sat == 0 and new_sat > 0:
                # Was UNSAT, now SAT → remove from list
                pos = unsat_pos[cc]
                # Swap with last element
                last = unsat_list[n_unsat - 1]
                unsat_list[pos] = last
                unsat_pos[last] = pos
                unsat_pos[cc] = -1
                n_unsat -= 1
            elif old_sat > 0 and new_sat == 0:
                # Was SAT, now UNSAT → add to list
                unsat_list[n_unsat] = cc
                unsat_pos[cc] = n_unsat
                n_unsat += 1

    return assignment, max_flips


# ══════════════════════════════════════════════════════════════════════
#  BSDTGravityV2 — same as H10b (fast version)
# ══════════════════════════════════════════════════════════════════════
class BSDTGravityV2:
    def __init__(self, n, clauses, mu_scale=0.1,
                 G_max=0.10, top_k_frac=0.1, gravity_start=0.2,
                 elite_repulsion=0.5, gravity_interval=20):
        self.n = n
        self.m = len(clauses)
        self.mu_scale = mu_scale
        self.G_max = G_max
        self.top_k_frac = top_k_frac
        self.gravity_start = gravity_start
        self.elite_repulsion = elite_repulsion
        self.gravity_interval = gravity_interval
        self.device = device

        vs_list = [vs for vs, ss in clauses]
        ss_list = [ss for vs, ss in clauses]
        self.vars_t  = torch.tensor(vs_list, dtype=torch.long,    device=device)
        self.signs_t = torch.tensor(ss_list, dtype=torch.float32, device=device)
        self.pos_mask = (self.signs_t > 0).long()

        # BPR clause format
        self.clauses_v = np.array(vs_list, dtype=np.int32)
        self.clauses_s = np.array(ss_list, dtype=np.int32)

    def _energy_core(self, s, mu_val, vars_t, signs_t):
        lit = s[:, vars_t] * signs_t.unsqueeze(0)
        e_sat = (torch.prod(1.0 - lit, dim=-1) / 8.0).sum(-1)
        if mu_val > 0:
            return e_sat + mu_val * ((1.0 - s * s) ** 2).sum(-1)
        return e_sat

    def find_best_particle(self, s):
        with torch.no_grad():
            x = (s > 0).long()
            lit_ok = (x[:, self.vars_t] == self.pos_mask.unsqueeze(0))
            n_sat = lit_ok.any(dim=2).sum(dim=1)
            best = n_sat.argmax()
            return best.item(), self.m - n_sat[best].item()

    def check_single(self, x):
        with torch.no_grad():
            n_sat = int((x[self.vars_t] == self.pos_mask).any(dim=1).sum())
            return n_sat == self.m, self.m - n_sat

    def get_confidence(self, s, best_idx):
        """Extract confidence = 1 - |s_v| from best particle's continuous position."""
        with torch.no_grad():
            return (1.0 - s[best_idx].abs()).cpu().numpy().astype(np.float64)

    def gravity_flow(self, steps=4000, particles=1000,
                     schedule='delay70', lr=0.02):
        n = self.n
        gi = self.gravity_interval
        s = torch.randn(particles, n, device=self.device) * 0.1
        s.requires_grad_(True)
        grav_step  = int(self.gravity_start * steps)
        delay_step = int(0.7 * steps)
        top_k = max(1, int(self.top_k_frac * particles))
        theta = None
        use_amp = (self.device.type == 'cuda')
        cached_target = None

        for step in range(steps):
            if step < delay_step:
                mu = 0.0
            else:
                t_l = (step - delay_step) / (steps - delay_step)
                mu = self.mu_scale * 0.5 * (1.0 - np.cos(np.pi * t_l))

            if use_amp:
                with torch.amp.autocast('cuda'):
                    e = self._energy_core(s, mu, self.vars_t, self.signs_t)
                    e_f32 = e.float()
            else:
                e_f32 = self._energy_core(s, mu, self.vars_t, self.signs_t)

            e_vals = e_f32.detach()
            e_f32.sum().backward()

            with torch.no_grad():
                if theta is None:
                    theta = float(e_vals.median()) + 1e-8
                damp = 1.0 / (1.0 + e_vals.unsqueeze(1) / theta)
                s.sub_(lr * damp * s.grad)

                if step >= grav_step and (step - grav_step) % gi == 0:
                    progress = (step - grav_step) / (steps - grav_step)
                    g = self.G_max * progress * progress * gi
                    _, top_idx = e_vals.topk(top_k, largest=False)
                    elite = s[top_idx]
                    d = torch.cdist(s, elite)
                    cached_target = elite[d.argmin(dim=1)]
                    if top_k > 1:
                        ed = d[top_idx]
                        ed.fill_diagonal_(float('inf'))
                        nn_e = ed.argmin(dim=1)
                        push = elite - elite[nn_e]
                        pn = push.norm(dim=1, keepdim=True).clamp_(min=1e-6)
                        s[top_idx] += (self.elite_repulsion * g) * (push / pn)
                    s.add_(g * (cached_target - s))
                elif step >= grav_step and cached_target is not None:
                    progress = (step - grav_step) / (steps - grav_step)
                    g = self.G_max * progress * progress
                    s.add_(g * (cached_target - s))

                s.clamp_(-1, 1)
                if (step + 1) % 200 == 0:
                    theta = float(e_vals.median()) + 1e-8

            s.requires_grad_(True)
            if s.grad is not None:
                s.grad.zero_()

        return s.detach()

# ── Compile + warmup ──────────────────────────────────────────────────
try:
    BSDTGravityV2._energy_core = torch.compile(BSDTGravityV2._energy_core)
    print('✓ torch.compile applied')
except Exception:
    print('⚠ torch.compile unavailable')

# Warmup BPR (triggers Numba compilation)
_conf = np.array([0.5, 0.3, 0.8], dtype=np.float64)
_ = bpr_repair(
    np.array([[0, 1, 2]], dtype=np.int32),
    np.array([[1, -1, 1]], dtype=np.int32),
    np.array([1, 0, 1], dtype=np.int32),
    _conf, max_flips=10
)

print('✓ BPR V2 + GravityV2 loaded')
print(f'  Device: {device}')
if device.type == 'cuda':
    print(f'  GPU:    {torch.cuda.get_device_name()}')
print()
print('  BPR innovations vs prior art:')
print('    1. Flip prob ∝ exp(-ΔE/T) × confidence(v)  [energy + continuous prior]')
print('    2. confidence(v) = 1-|s_v| from BSDT positions [NO other SLS has this]')
print('    3. Incremental unsat list — O(1) clause pick, O(deg(v)) flip update')
print('    4. Cosine temperature anneal T: 0.5 → 0.01')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# H11 Experiment: GravityV2 + BPR vs GravityV2 + WalkSAT
# ══════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt

ALPHAS   = [3.8, 4.0, 4.2]
NS       = [500, 750, 1000]
N_INST   = 50
PARTICLES = 1000
STEPS    = 4000

# Baselines
h10b = {
    (3.8, 500): 98.0, (3.8, 750): 98.0, (3.8, 1000): 96.0,
    (4.0, 500): 72.0, (4.0, 750): 60.0, (4.0, 1000): 30.0,
    (4.2, 500): 10.0, (4.2, 750):  6.0, (4.2, 1000):  0.0,
}
h9b = {
    (3.8, 500): 94.0, (3.8, 750): 86.0, (3.8, 1000): 72.0,
    (4.0, 500): 44.0, (4.0, 750): 26.0, (4.0, 1000):  4.0,
    (4.2, 500):  8.0, (4.2, 750):  0.0, (4.2, 1000):  0.0,
}

results = {}
print('=' * 88)
print('H11 — GravityV2 + BPR (fully original, no WalkSAT)')
print('=' * 88)
print(f"  {'α':>5} | {'n':>5} | {'S1':>6} | {'Final':>6} | {'Viols':>5} | "
      f"{'Flips':>7} | {'BPR':>7} | {'H10b':>5} | {'Δ':>6} | Time")
print('  ' + '-' * 82)

for alpha in ALPHAS:
    for n_var in NS:
        m = int(alpha * n_var)
        t0 = time.time()
        s1 = bpr_ok = bpr_tried = 0
        tot_v = tot_f = fc = 0

        for inst in range(N_INST):
            clauses = generate_3sat_instance(n_var, m)
            eng = BSDTGravityV2(n_var, clauses)
            sf = eng.gravity_flow(steps=STEPS, particles=PARTICLES)

            best_idx, viols = eng.find_best_particle(sf)

            if viols == 0:
                s1 += 1
            else:
                tot_v += viols; fc += 1
                x_np = (sf[best_idx] > 0).cpu().numpy().astype(np.int32)
                # Extract confidence from continuous positions
                conf = eng.get_confidence(sf, best_idx)
                # BPR repair — YOUR method
                sol, flips = bpr_repair(
                    eng.clauses_v, eng.clauses_s, x_np.copy(), conf,
                    max_flips=100000, T_init=0.5, T_min=0.01, p_random=0.1
                )
                tot_f += flips; bpr_tried += 1
                sol_t = torch.tensor(sol, dtype=torch.long, device=device)
                sat, _ = eng.check_single(sol_t)
                if sat:
                    bpr_ok += 1

            if (inst + 1) % 10 == 0:
                el = time.time() - t0
                cur_ok = s1 + bpr_ok
                print(f'    α={alpha}, n={n_var}: instance {inst+1}/{N_INST} done, '
                      f'{cur_ok} solved so far '
                      f'({el:.0f}s, ~{el/(inst+1)*N_INST:.0f}s total)')

        elapsed = time.time() - t0
        final = (s1 + bpr_ok) / N_INST * 100
        avg_v = tot_v / max(fc, 1)
        avg_f = tot_f // max(bpr_tried, 1)
        ws_ref = h10b[(alpha, n_var)]
        delta = final - ws_ref
        tag = '★' if final >= 95 else ('▲' if delta > 0 else
              ('=' if abs(delta) < 1 else '▼'))

        results[(alpha, n_var)] = {
            's1': s1 / N_INST * 100, 'final': final,
            'viols': avg_v, 'flips': avg_f,
            'bpr': f'{bpr_ok}/{bpr_tried}', 'h10b': ws_ref, 'delta': delta
        }
        print(f'  {alpha:5.1f} | {n_var:5d} | {s1/N_INST*100:5.1f}% | '
              f'{final:5.1f}% | {avg_v:5.1f} | {avg_f:7d} | '
              f'{bpr_ok:3d}/{bpr_tried:<3d} | {ws_ref:4.0f}% | {delta:+5.1f}% | '
              f'{elapsed:4.0f}s {tag}')
    print('  ' + '-' * 82)

print('\nGenerating charts...\n')

# ── 3-way comparison ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
w = 0.25
for i, n_var in enumerate(NS):
    ax = axes[i]
    base  = [h9b[(a, n_var)]  for a in ALPHAS]
    ws    = [h10b[(a, n_var)] for a in ALPHAS]
    bpr_v = [results[(a, n_var)]['final'] for a in ALPHAS]
    x = np.arange(len(ALPHAS))

    ax.bar(x - w, base, w, label='H9b (no gravity+WS)', color='#e74c3c', alpha=0.8)
    ax.bar(x,     ws,   w, label='H10b (gravity+WS)',   color='#3498db', alpha=0.8)
    ax.bar(x + w, bpr_v, w, label='H11 (gravity+BPR)',  color='#2ecc71', alpha=0.8)

    ax.set_xticks(list(x))
    ax.set_xticklabels([str(a) for a in ALPHAS])
    ax.set_xlabel('α (clause ratio)')
    ax.set_ylabel('Solve Rate %')
    ax.set_title(f'n = {n_var}')
    ax.set_ylim(0, 105)
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('H11: Fully Original Pipeline (GravityV2 + BPR) — No WalkSAT',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('h11_bpr_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: h11_bpr_results.png')